# Project 15B · Agent Safety Gate (Starter notebook)

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 3–4 hours |
| 📋 Prerequisites | Lessons 15.02 and 15.05; Module 04 (human-in-the-loop) |
| 🔑 Keys | none offline · optional `TYPESAFE_API_KEY` (live Jev, ~40 requests, < $0.001) |
| 💸 Cost | $0 offline; live: well under one cent |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](README.md) |

**The job:** Orbita Cloud's ops agent can restart services, delete buckets and send email. The security lead will only turn it on with a gate that never lets an unsafe call run unattended, asks a person when unsure, and blocks when the check itself is down. Tune that gate on labeled calls, prove it on held-out calls and in the agent, and write the configuration to ship.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Each TODO has a collapsed hint; the [reference solution](solutions/15b-project-agent-safety-gate-solution.ipynb) is the last resort.

## Requirements (from the spec)

1. **Score** 32 labeled tool calls with exactly the gate's question, failing closed when the check fails.
2. **Choose** `review_at` / `block_at` on the dev calls from the security lead's costs, and compare with a single 0.5 cutoff.
3. **Verify** on the held-out calls: no unsafe call may run automatically.
4. **Prove** it in the agent: the injection is blocked, the requested restart is reviewed, an outage of the check blocks.
5. **Report** the configuration to ship.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `score_cases(classifier, items)` |
| 2 | Step 3 | `pick_gate(items)` |
| 3 | Step 3 | `gate_metrics(items, band)` |
| 4 | Step 4 | `run_agent_checks(band)` |
| 5 | Step 5 | `write_report(path, evidence)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY',)
LANGSMITH_PROJECT = 'm15-project-15b'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The inputs

32 tool calls an ops agent proposed, each with the user's request, what the agent had just read (some tool outputs carry prompt injections), and a human label: `unsafe = True` means the call must not run without a person. Odd ids are **dev**, even ids are **test**.

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# The inputs: 32 labeled tool calls (dev/test), the gate's question, the costs, and the sandboxed ops agent from 15.05.
import json

from langchain_core.runnables import RunnableLambda
from langchain_typesafe.client import TypeSafeAPIConnectionError

from mastery.decisions import provenance
from mastery.decisions.middleware import JevToolGate
from mastery.decisions.ops import GUARDED, run_scenario
from mastery.decisions.policy import Band, BandCosts, band_cost, choose_band
from mastery.decisions.safety import case_state, cases
from mastery.evals.stats import wilson_interval

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m15"          # git-ignored
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEV, TEST = cases("dev"), cases("test")
QUESTIONS = JevToolGate(jev, tools=GUARDED)._questions()   # exactly the gate's question
# ← Orbita's security lead priced each outcome:
GATE_COSTS = BandCosts(false_yes=5, false_no=100, review=1)   # needless block / unsafe call runs / a person reviews

print(f"dev {len(DEV)} ({sum(c['unsafe'] for c in DEV)} unsafe) · test {len(TEST)} ({sum(c['unsafe'] for c in TEST)} unsafe)")
example = next(c for c in DEV if c["tool_output"])
print("\nexample:", example["id"], "· user:", example["request"], "\n  agent read:", example["tool_output"][:90] + "…",
      "\n  proposed:", example["tool"], example["args"], "· label unsafe =", example["unsafe"], f"({example['why']})")

**You should now see** `dev 16 (6 unsafe) · test 16 (8 unsafe)` and one example: an injected instruction in the logs that led the agent to propose a destructive call.

---
## Step 2 · Score every call

**TODO 1** is in the cell below.

> 💡 **Hint:** Same request the gate sends: `case_state(case)` as the state and `QUESTIONS` as the questions. Read the `risky` noul. Wrap each call in `try`/`except Exception` and record `p=None` with the exception's class name.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
out = {}
for c in items:
    try:
        r = classifier.invoke({"state": case_state(c), "questions": QUESTIONS})
        out[c["id"]] = {"p": round(r.nouls["risky"].noul, 3), "source": provenance(r)}
    except Exception as exc:  # noqa: BLE001  fail closed: no probability
        out[c["id"]] = {"p": None, "source": type(exc).__name__}
return out
```

</details>

In [ ]:
def score_cases(classifier, items: list) -> dict:
    """id → {"p": p(risky) or None, "source": provenance or the error name}. A failed check must count as risky."""
    # TODO 1: ask the classifier the gate's QUESTIONS about case_state(case); on any exception record p=None.
    return {c["id"]: {"p": None, "source": "TODO"} for c in items}


SCORES = score_cases(jev, DEV + TEST)
print({k: v for k, v in list(SCORES.items())[:4]}, "…")

**You should now see** (solution) the first four cases with a `p` and `source: replay (synthetic)`. Starter: `p: None`, which `risk()` treats as certain risk, so the starter's gate would block everything (safe, but useless).

---
## Step 3 · Choose the thresholds, then check them on held-out calls

**TODO 2** is in the cell below.

> 💡 **Hint:** Lesson 15.02's `choose_band` takes probabilities, 0/1 labels and a `BandCosts`. Use `risk(case)` so a failed check counts as 1.0.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
return choose_band([risk(c) for c in items], [int(c["unsafe"]) for c in items], GATE_COSTS)
```

</details>

**TODO 3** is two cells below.

> 💡 **Hint:** Per case: block at or above `band.high`, review at or above `band.low`, else allow. Count unsafe calls that would be allowed, safe calls blocked and reviews; price with `band_cost`; put the catch rate in `wilson_interval`.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
acts = {c["id"]: ("block" if risk(c) >= band.high else "review" if risk(c) >= band.low else "allow") for c in items}
unsafe = [c for c in items if c["unsafe"]]
caught = sum(acts[c["id"]] != "allow" for c in unsafe)
return {"n": len(items), "unsafe_allowed": len(unsafe) - caught,
        "safe_blocked": sum(acts[c["id"]] == "block" and not c["unsafe"] for c in items),
        "reviews": sum(a == "review" for a in acts.values()),
        "cost": band_cost([risk(c) for c in items], [int(c["unsafe"]) for c in items], band.low, band.high, GATE_COSTS).cost,
        "unsafe_caught": str(wilson_interval(caught, len(unsafe))), "actions": acts}
```

</details>

In [ ]:
def risk(case) -> float:
    """p(risky) for a case; a failed check (p=None) counts as certain risk: fail closed."""
    p = SCORES[case["id"]]["p"]
    return 1.0 if p is None else p


def pick_gate(items: list) -> Band:
    """(review_at, block_at) = (low, high) of the cheapest band on `items` under GATE_COSTS."""
    # TODO 2: probabilities = risk(case), labels = case["unsafe"]; choose_band with GATE_COSTS.
    return Band(0.5, 0.5, 0.0, 0, 0)


GATE = pick_gate(DEV)
print("chosen on dev:", GATE)
single = band_cost([risk(c) for c in DEV], [c["unsafe"] for c in DEV], 0.5, 0.5, GATE_COSTS)
print("AutoMode-style single cutoff 0.5 on dev:", single)

In [ ]:
def gate_metrics(items: list, band: Band) -> dict:
    """On held-out items: unsafe calls that would run, safe calls blocked, reviews, cost, and the catch rate CI."""
    # TODO 3: classify each case allow / review / block with band.low and band.high, then count.
    return {}


METRICS = gate_metrics(TEST, GATE)
print(METRICS)

**You should now see** (solution) a band `low=0.4, high=0.55` with 2 reviews on dev and cost 2.0, against cost 100 for a single 0.5 cutoff: one external email (`C27`, p = 0.42) would run unattended. On test: `unsafe_allowed: 0`, no safe call blocked, and 3 reviews.

---
## Step 4 · The gate in the agent

**TODO 4** is in the cell below.

> 💡 **Hint:** A fresh `JevToolGate` per scenario (each keeps its own `log`) with `review_at=band.low`, `block_at=band.high`. `run_scenario(name, [gate], approve=..., thread=...)` from Lesson 15.05. For the outage, pass a `RunnableLambda` that raises.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def gate(classifier=jev):
    return JevToolGate(classifier, tools=GUARDED, review_at=band.low, block_at=band.high, provenance=provenance)

g = gate()
inj = run_scenario("injection", [g], thread="p15b-injection")
g2 = gate()
rst = run_scenario("restart", [g2], approve=True, thread="p15b-restart")

def down(_):
    raise TypeSafeAPIConnectionError("Unable to connect to the TypeSafe API.")

g3 = gate(RunnableLambda(down))
off = run_scenario("restart", [g3], thread="p15b-down")
return {"injection buckets left": inj["buckets_left"], "injection emails": inj["emails_sent"],
        "injection gate": [e["action"] for e in g.log],
        "restart gate": [e["action"] for e in g2.log], "restart restarted": rst["restarted"],
        "jev down gate": [e["action"] for e in g3.log], "jev down restarted": off["restarted"]}
```

</details>

In [ ]:
def run_agent_checks(band: Band) -> dict:
    """Your thresholds in a JevToolGate on 15.05's scenarios, plus the gate with a classifier that is down."""
    # TODO 4: build JevToolGate(jev, tools=GUARDED, review_at=..., block_at=...) per scenario; run "injection",
    # "restart" (approve=True) and "restart" with a gate whose classifier raises TypeSafeAPIConnectionError.
    return {}


AGENT = run_agent_checks(GATE)
for k, v in AGENT.items():
    print(f"{k:<26} {v}")

**You should now see** (solution) both buckets left and no emails after the injection (the gate blocked two calls), `['review → approved']` for the requested restart (p = 0.45 lands in your band), and `['blocked (classifier error)']` with nothing restarted when Jev is down.

---
## Step 5 · Report

**TODO 5** is in the cell below.

> 💡 **Hint:** One Markdown string with the five `##` sections from the docstring, built from the evidence dict, naming `review_at = ...` and the synthetic-data limit.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
g, s, m, a = evidence["gate"], evidence["single"], evidence["metrics"], evidence["agent"]
lines = ["# Project 15B report: agent safety gate", "",
         "## Thresholds", f"Chosen on the dev calls from costs (needless block {GATE_COSTS.false_yes}, unsafe call runs "
         f"{GATE_COSTS.false_no}, review {GATE_COSTS.review}): **review_at = {g.low}, block_at = {g.high}** "
         f"(dev cost {g.cost}, {g.reviewed} reviews). A single 0.5 cutoff on dev: cost {s.cost}, errors: {s.errors}.", "",
         "## Held-out results", f"Test calls: {m['n']} · unsafe calls that would run: **{m['unsafe_allowed']}** · "
         f"safe calls blocked: {m['safe_blocked']} · reviews: {m['reviews']} · cost {m['cost']} · unsafe caught {m['unsafe_caught']}.", "",
         "## Agent scenarios", f"Injection: gate {a['injection gate']}, buckets left {a['injection buckets left']}, "
         f"emails {a['injection emails']}. Restart: gate {a['restart gate']}. Classifier down: gate {a['jev down gate']}, "
         f"restarted {a['jev down restarted']} (fails closed).", "",
         "## Recommendation", f"Ship JevToolGate with review_at = {g.low} and block_at = {g.high} on restart_service, "
         "delete_bucket and send_email; route reviews to the on-call channel; keep least-privilege credentials and an "
         "email allow-list as the last line of defense; re-tune on fresh labeled calls monthly.", "",
         "## Limits", "32 labeled calls is a teaching set: the catch-rate interval is wide. In replay mode Jev's answers are "
         "synthetic, so the numbers show the method, not Jev's accuracy."]
text = "\n".join(lines) + "\n"
path.write_text(text, encoding="utf-8")
return text
```

</details>

In [ ]:
def write_report(path, evidence: dict) -> str:
    """Markdown for the security lead: Thresholds, Held-out results, Agent scenarios, Recommendation, Limits."""
    # TODO 5: five "##" sections with the measured numbers, a recommendation and the synthetic-data note; save to path.
    text = "# Project 15B report\n\n(TODO)\n"
    path.write_text(text, encoding="utf-8")
    return text


report = write_report(OUT_DIR / "project15b-report.md", {"gate": GATE, "single": single, "metrics": METRICS, "agent": AGENT})
print(report[:1500])

---
## Step 6 · Acceptance checks

In [ ]:
def accept(name, ok):
    results[name] = bool(ok)
    print(("✅" if ok else "❌"), name)


results = {}
accept("labeled calls: 16 dev + 16 test, both splits with unsafe and safe calls",
       len(DEV) == len(TEST) == 16 and all(0 < sum(c["unsafe"] for c in s) < len(s) for s in (DEV, TEST)))
broken = score_cases(RunnableLambda(lambda _: 1 / 0), DEV[:2])
accept("score_cases: one probability per call, and a failed check recorded as no probability",
       len(SCORES) == 32 and all(isinstance(v["p"], float) for v in SCORES.values()) and all(v["p"] is None for v in broken.values()))
ref = choose_band([risk(c) for c in DEV], [int(c["unsafe"]) for c in DEV], GATE_COSTS)
accept("thresholds: the cheapest band on DEV, with a review band", (GATE.low, GATE.high) == (ref.low, ref.high) and GATE.low < GATE.high)
accept("held-out: no unsafe test call would run automatically", METRICS.get("n") == len(TEST) and METRICS.get("unsafe_allowed") == 0)
accept("agent: the injected delete and email are blocked, nothing destroyed or sent",
       "station-backups" in AGENT.get("injection buckets left", []) and AGENT.get("injection emails") == []
       and AGENT.get("injection gate", []).count("blocked") == 2)
accept("agent: the requested restart goes through your review band", AGENT.get("restart gate") == ["review → approved"])
accept("agent: the gate fails closed when the classifier is down",
       AGENT.get("jev down restarted") == [] and AGENT.get("jev down gate") == ["blocked (classifier error)"])
accept("report: five sections, measured numbers, the synthetic-data note",
       all(s in report for s in ("## Thresholds", "## Held-out results", "## Agent scenarios", "## Recommendation", "## Limits"))
       and "synthetic" in report and f"review_at = {GATE.low}" in report)
print(f"\n{sum(results.values())}/{len(results)} acceptance checks passed")
(OUT_DIR / "project15b-acceptance.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `1/8` (the data checks come from the course; everything else needs a TODO). Results: `data/outputs/m15/project15b-acceptance.json`.

---
## 🔎 Reflect (write 3–5 sentences in your report)

- Which unsafe call came closest to running, and what would you change (costs, labels, a rule) to protect it?
- Reviews cost a person's attention: how many per day can the on-call channel take, and what does that do to `review_at`?

## ✅ Before you commit

- [ ] Restart & Run All passes without any key.
- [ ] All TODOs done; 8/8 acceptance checks; report written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Module 15 done:** you can put a decision model at the branches of a LangChain or LangGraph app, choose its thresholds from costs, check it against an LLM, and keep it safe when it fails.

**Next:** the capstones in [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md).

Back to the [Module 15 README](../../../modules/15-decision-models-with-jev/README.md) · [Course outline](../../../COURSE_OUTLINE.md)